# 00. Dasar PyTorch melalui teks

Setelah modul ini, Anda dapat membedakan token, indeks token, dan embedding. Anda juga dapat membaca bentuk tensor, melakukan perkalian matriks, serta memeriksa gradien.

Kerangka kerja pembelajaran mendalam menyediakan tensor, diferensiasi otomatis, lapisan model, dan pengoptimal. PyTorch memungkinkan operasi tersebut ditulis sebagai kode Python yang bisa diperiksa langkah demi langkah.

**Prasyarat:** dasar Python, daftar, fungsi, dan perkalian matriks sederhana.

**Pola belajar:** baca penjelasan, prediksi bentuk keluaran, jalankan kode, lalu ubah satu hal.

Contoh ulasan dalam paket ini merupakan data sintetis untuk mempelajari mekanisme. Metriknya tidak mewakili kinerja pada ulasan nyata.

## Penyiapan

Instal dependensi melalui petunjuk README sebelum menjalankan seluruh sel. Setiap notebook dapat dimulai dengan kernel baru. GPU bersifat opsional. Semua operasi tensor yang berinteraksi harus berada pada perangkat yang sesuai.

In [1]:
from pathlib import Path
import sys
# Lokal: buka dari root repo, folder nlp, atau nlp/notebooks.
# Colab: ambil paket kursus jika belum tersedia.
candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for base in candidates for p in (base, base / "nlp")
             if (p / "nlp_course").is_dir()), None)
if ROOT is None and "google.colab" in sys.modules:
    import subprocess
    target = Path("/content/pytorch-deep-learning-nlp")
    if not target.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none",
                        "--sparse", "--branch", "nlp-learning-path",
                        "https://github.com/FeliksMakarios/pytorch-deep-learning.git",
                        str(target)], check=True)
        subprocess.run(["git", "sparse-checkout", "set", "nlp"], cwd=target, check=True)
    ROOT = target / "nlp"
if ROOT is None or not (ROOT / "nlp_course").is_dir():
    raise RuntimeError("Folder nlp_course tidak ditemukan. Ikuti petunjuk README nlp.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import torch
from torch import nn
from nlp_course.data import tokenize, build_vocab, encode, read_rows, loaders, collate_batch
from nlp_course.models import MeanClassifier, RecurrentClassifier, TinyTransformer
from nlp_course.engine import seed_all, fit, run_epoch, metrics, save_mean, load_mean, predict
seed_all(42)
torch.set_num_threads(1)
device = "cuda" if torch.cuda.is_available() else "cpu"
ARTIFACTS = ROOT / "artifacts"
ARTIFACTS.mkdir(exist_ok=True)
print("PyTorch:", torch.__version__, "Perangkat:", device)

PyTorch: 2.14.0+cu130 Perangkat: cpu


## 1. Dari kalimat ke token

Komputer memerlukan representasi numerik. Tokenisasi membagi teks menjadi unit. Di sini unitnya berupa kata dan tanda baca, belum subword seperti pada BERT. Negasi seperti `tidak` tetap dipertahankan karena dapat mengubah makna sentimen.

In [2]:
texts = ["buku ini baik", "buku ini tidak baik"]
for text in texts:
    print(text, "->", tokenize(text))
vocab = build_vocab(texts)
print(vocab)

buku ini baik -> ['buku', 'ini', 'baik']
buku ini tidak baik -> ['buku', 'ini', 'tidak', 'baik']
{'<pad>': 0, '<unk>': 1, 'baik': 2, 'buku': 3, 'ini': 4, 'tidak': 5}


## 2. Indeks merupakan alamat, bukan tingkat makna

Indeks token bersifat kategoris. Indeks 6 tidak berarti dua kali lebih positif daripada indeks 3. `torch.long` digunakan untuk indeks. Angka pecahan digunakan untuk bobot model.

In [3]:
ids = torch.tensor([encode(t, vocab) for t in [texts[1]]], dtype=torch.long)
print(ids, ids.shape, ids.dtype, ids.device)
print("Kalimat pertama:", ids[0])
print("Token pertama:", ids[0, 0].item())

tensor([[3, 4, 5, 2]]) torch.Size([1, 4]) torch.int64 cpu
Kalimat pertama: tensor([3, 4, 5, 2])
Token pertama: 3


## 3. Embedding sebagai tabel yang dipelajari

Tabel embedding berbentuk `[V, D]`, dengan V ukuran kosakata dan D banyaknya fitur. Input `[B, T]` menghasilkan `[B, T, D]`. Bobot awal belum memiliki makna linguistik yang andal. Makna berkembang melalui tugas pelatihan.

In [4]:
embedding = nn.Embedding(len(vocab), 3, padding_idx=0)
x = embedding(ids)
print("Tabel:", embedding.weight.shape)
print("Hasil:", x.shape)
print(x)
assert x.shape == (1, 4, 3)

Tabel: torch.Size([6, 3])
Hasil: torch.Size([1, 4, 3])
tensor([[[-0.6866,  0.6105,  1.3347],
         [-0.2316,  0.0418, -0.2516],
         [ 0.8599, -0.3097, -0.3957],
         [ 0.8008,  1.6806,  0.3559]]], grad_fn=<EmbeddingBackward0>)


## 4. Padding dan dimensi batch

Kalimat berbeda panjang perlu disatukan dalam batch. Padding menyamakan panjang, sedangkan mask menandai posisi yang nyata. Rata-rata harus dibagi jumlah token nyata, bukan panjang setelah padding.

In [5]:
batch = [(torch.tensor(encode(t,vocab)), label) for t,label in zip(texts, [1,0])]
# 1 = positif, 0 = negatif, konsisten dengan seluruh kursus.
ids, lengths, labels = collate_batch(batch)
x = embedding(ids)
mask = ids.ne(0).unsqueeze(-1)
pooled = (x * mask).sum(dim=1) / mask.sum(dim=1)
print("Indeks:", ids)
print("Panjang:", lengths)
print("Mask:", mask.squeeze(-1))
print("Vektor kalimat:", pooled, pooled.shape)

Indeks: tensor([[3, 4, 2, 0],
        [3, 4, 5, 2]])
Panjang: tensor([3, 4])
Mask: tensor([[ True,  True,  True, False],
        [ True,  True,  True,  True]])
Vektor kalimat: tensor([[-0.0391,  0.7776,  0.4797],
        [ 0.1856,  0.5058,  0.2608]], grad_fn=<DivBackward0>) torch.Size([2, 3])


## 5. Perkalian matriks dan transpose

Vektor kalimat `[B, D]` dikalikan bobot `[D, C]` menghasilkan skor `[B, C]`. `nn.Linear` menyimpan bobot sebagai `[C, D]`, sehingga operasi setaranya menggunakan transpose. Softmax mengubah skor menjadi distribusi, tetapi loss klasifikasi PyTorch menerima skor mentah.

In [6]:
head = nn.Linear(3, 2)
manual = pooled @ head.weight.T + head.bias
logits = head(pooled)
print("Bobot:", head.weight.shape, "Logits:", logits.shape)
print(logits)
print(logits.softmax(-1))
assert torch.allclose(manual, logits)

Bobot: torch.Size([2, 3]) Logits: torch.Size([2, 2])
tensor([[ 0.4831, -0.2131],
        [ 0.3242, -0.0856]], grad_fn=<AddmmBackward0>)
tensor([[0.6673, 0.3327],
        [0.6010, 0.3990]], grad_fn=<SoftmaxBackward0>)


## 6. Bentuk tensor dan broadcasting

`unsqueeze` menambah sumbu berukuran satu. `squeeze` menghapus sumbu tersebut. Gunakan argumen dimensi agar batch berukuran satu tidak ikut terhapus. Broadcasting memperluas dimensi yang kompatibel tanpa harus menyalin data secara manual.

In [7]:
a = torch.arange(6, dtype=torch.float32).reshape(2, 3)
print(a, a.T, sep="\n")
print("Tambah dimensi:", a.unsqueeze(1).shape)
print("Tambah bias:", (a + torch.tensor([1.,2.,3.])))
print("Rerata tiap baris:", a.mean(1))
print("Gabung:", torch.cat([a,a],dim=0).shape)
print("Tumpuk:", torch.stack([a,a]).shape)

tensor([[0., 1., 2.],
        [3., 4., 5.]])
tensor([[0., 3.],
        [1., 4.],
        [2., 5.]])
Tambah dimensi: torch.Size([2, 1, 3])
Tambah bias: tensor([[1., 3., 5.],
        [4., 6., 8.]])
Rerata tiap baris: tensor([1., 4.])
Gabung: torch.Size([4, 3])
Tumpuk: torch.Size([2, 2, 3])


## 7. Gradien dan pembaruan bobot

Diferensiasi otomatis menghitung turunan loss terhadap bobot. `backward()` mengakumulasi gradien. Kosongkan gradien sebelum pembaruan berikutnya. Di sini kita melihat seluruh vektor embedding satu token sebelum dan setelah SGD.

In [8]:
loss = nn.CrossEntropyLoss()(logits, labels)
optimizer = torch.optim.SGD(list(embedding.parameters()) + list(head.parameters()), lr=0.1)
word_id = vocab["baik"]
before = embedding.weight[word_id].detach().clone()
optimizer.zero_grad()
loss.backward()
grad = embedding.weight.grad[word_id].clone()
print("Sebelum:", before)
print("Gradien:", grad)
print("Perubahan -lr*grad:", -0.1*grad)
optimizer.step()
after = embedding.weight[word_id].detach()
print("Sesudah:", after)
assert torch.allclose(after, before - 0.1*grad)
assert torch.count_nonzero(embedding.weight.grad[0]) == 0

Sebelum: tensor([0.8008, 1.6806, 0.3559])
Gradien: tensor([-0.0401,  0.0180,  0.0167])
Perubahan -lr*grad: tensor([ 0.0040, -0.0018, -0.0017])
Sesudah: tensor([0.8048, 1.6788, 0.3542])


## 8. Perangkat, reproduksibilitas, dan NumPy

Model dan indeks token dipindahkan bersama. Gunakan `detach().cpu().numpy()` untuk mengubah tensor bergradien menjadi array NumPy. Seed membantu mengulang eksperimen, tetapi hasil lintas perangkat dan versi tidak selalu identik.

In [9]:
embedding = embedding.to(device)
print(embedding(ids.to(device)).shape)
print(after.detach().cpu().numpy())
seed_all(7)
r1 = torch.rand(2)
seed_all(7)
assert torch.equal(r1, torch.rand(2))

torch.Size([2, 4, 3])
[0.80481267 1.678819   0.35419074]


## Latihan mandiri

1. Ubah dimensi embedding dari 3 menjadi 5. Bentuk apa yang berubah?
2. Mengapa token PAD tidak boleh ikut penyebut rata-rata?
3. Hitung pembaruan jika bobot [1,2], gradien [0.4,-0.2], dan laju belajar 0.5.
4. Apa perbedaan indeks token, embedding, dan probabilitas kelas?

## Pembahasan latihan

1. Tabel menjadi `[V,5]`, keluaran embedding `[B,T,5]`, vektor kalimat `[B,5]`. Masukan Linear juga harus 5.
2. Padding akan mengecilkan vektor kalimat pendek secara tidak semestinya.
3. `[1,2] - 0.5 × [0.4,-0.2] = [0.8,2.1]`.
4. Indeks memilih baris, embedding menyimpan fitur yang dipelajari, probabilitas merangkum skor prediksi.

## Penghubung ke materi berikutnya

Kita sudah memiliki operasi satu batch. Modul 01 merangkainya menjadi siklus pelatihan dengan data validasi dan uji.

### Rujukan
- [Dokumentasi PyTorch](https://docs.pytorch.org/docs/stable/index.html)
- [Sumber Embedding](https://github.com/pytorch/pytorch/blob/main/torch/nn/modules/sparse.py)
- [Sumber Transformer](https://github.com/pytorch/pytorch/blob/main/torch/nn/modules/transformer.py)
- [Kursus sumber dan struktur awal](https://github.com/mrdbourke/pytorch-deep-learning)

Materi ini ditulis sebagai jalur NLP mandiri. Penjelasan dan contoh NLP bukan terjemahan resmi kursus sumber.